# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook demonstrates how to load, explore, and process the FAIR² dataset using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via a Croissant schema URL.

In [ ]:
# Ensure mlcroissant is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the dataset Croissant schema URL
croissant_url = "https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json"

# Load the dataset metadata
dataset = mlc.Dataset(croissant_url)
print(f"Dataset name: {dataset.metadata.name}\nDescription: {dataset.metadata.description}")

## 2. Data Overview
Review available record sets, their fields and field `@id`s.

We'll enumerate all record sets and their fields. All entities are referenced by their `@id`.

In [ ]:
# List all record sets and their fields, referencing by @id
record_sets = dataset.metadata.record_sets
for rs in record_sets:
    print(f"RecordSet name: {rs.name}")
    print(f"  @id: {rs.id}")
    print(f"  Description: {rs.description}")
    print("  Fields:")
    for field in rs.fields:
        print(f"    - {field.name} (@id: {field.id}, dtype: {field.data_type})")
    print()

## 3. Data Extraction
Load data from a specific record set into a DataFrame for analysis. Use the record set and field `@id`s from the overview.

We will extract all record sets into DataFrames for further processing.

In [ ]:
# Extract data from each record set by @id
dataframes = {}
for rs in dataset.metadata.record_sets:
    print(f"Loading data for RecordSet {rs.name} (@id: {rs.id}) ...")
    records = list(dataset.records(record_set=rs.id))
    df = pd.DataFrame(records)
    dataframes[rs.id] = df
    print(f"DataFrame columns: {df.columns.tolist()}")
    print(f"Sample data:")
    print(df.head(), "\n---\n")

## 4. Exploratory Data Analysis (EDA)
Apply common data processing steps, such as filtering records based on specific criteria, normalizing numeric fields, and grouping data.

We select a record set and one of its numeric fields by `@id` for basic EDA. Adjust the variables below according to the actual field ids and data overview.

In [ ]:
# Example EDA on the main protein abundance record set
# Update the following variables based on actual @id values obtained above:
record_set_id = None  # e.g., 'cr:MainProteinAbundance' or the detected @id
numeric_field_id = None  # e.g., '@id' for the abundance/coverage/molecular weight field
group_field_id = None  # e.g., '@id' for a categorical field (sample id, uniprot accession, etc.)

# User must update these values based on the printed overview above
if not record_set_id:
    print("Please set the 'record_set_id', 'numeric_field_id', and optionally 'group_field_id' to proceed.")
else:
    df = dataframes[record_set_id]
    # Filter numeric field > threshold
    threshold = 10
    print(f"Filtering records with {numeric_field_id} > {threshold}")
    filtered_df = df[df[numeric_field_id] > threshold].copy()
    print(filtered_df.head())

    # Normalize the numeric field
    normalized_col = f"{numeric_field_id}_normalized"
    filtered_df[normalized_col] = (filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()) / filtered_df[numeric_field_id].std()
    print(f"Normalized {numeric_field_id} for filtered records:")
    print(filtered_df[[numeric_field_id, normalized_col]].head())

    # Example group by field (if exists)
    if group_field_id and group_field_id in filtered_df.columns:
        grouped_df = filtered_df.groupby(group_field_id)[numeric_field_id].mean().reset_index()
        print(f"Mean {numeric_field_id} grouped by {group_field_id}:")
        print(grouped_df.head())

## 5. Visualization
Visualize data distributions or relationships between fields in the dataset. Update variables as needed to explore the relevant numeric fields or categories.

In [ ]:
# Example visualization (requires matplotlib)
import matplotlib.pyplot as plt
import seaborn as sns

# Only run if record_set_id and numeric_field_id are set
if not record_set_id or not numeric_field_id:
    print("Set 'record_set_id' and 'numeric_field_id' for visualization.")
else:
    df = dataframes[record_set_id]
    plt.figure(figsize=(8, 6))
    sns.histplot(df[numeric_field_id].dropna(), bins=30, kde=True)
    plt.title(f"Distribution of {numeric_field_id} in Record Set {record_set_id}")
    plt.xlabel(numeric_field_id)
    plt.ylabel("Frequency")
    plt.show()

## 6. Conclusion
This notebook demonstrated how to load and process a FAIR² Croissant dataset using `mlcroissant`, referencing each entity by its `@id`.

You may now extend this notebook with additional domain-specific analysis or visualizations tailored to your research questions.